# 06: Query Difficulty Analysis
## Thesis: Understanding Why Some Queries Are Harder Than Others

**Objective**: Analyze query difficulty in HNSW-based ANN search and understand why adaptive methods help.

### Key Questions:
1. Why are some queries easy and others hard?
2. How does query difficulty affect recall and latency?
3. Why do adaptive methods (PiP, DARTH, Ada-ef) help?

### Key Concepts:
- **Query Difficulty**: How hard it is to find true nearest neighbors with limited search
- **Intrinsic Difficulty**: Related to local density, query position in data space
- **Search Difficulty**: Related to ef settings, graph structure
- **Easy Queries**: Well-separated neighbors, clear proximity structure
- **Hard Queries**: Dense neighborhoods, boundary cases, outliers

---

## 1. Imports and Path Setup

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJECT_ROOT = Path('/Users/Damian/approximate-nearest-neighbor-graphs')
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_CSV = PROJECT_ROOT / 'results_csv'
PLOT_RESULTS = PROJECT_ROOT / 'plot_results'
DATASETS_DIR = PROJECT_ROOT / 'Datasets'

RESULTS_CSV.mkdir(exist_ok=True)
PLOT_RESULTS.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results CSV dir: {RESULTS_CSV}")
print(f"Plot results dir: {PLOT_RESULTS}")

---

## 2. Load Dataset and Ground Truth

In [ ]:
from utils.read_files import read_fvecs, read_ivecs

DATASET_NAME = 'siftsmall'

BASE_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_base.fvecs'
QUERY_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_query.fvecs'
GT_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_groundtruth.ivecs'

print(f"Loading dataset: {DATASET_NAME}")
xb = read_fvecs(str(BASE_FILE))
xq = read_fvecs(str(QUERY_FILE))
I_gt = read_ivecs(str(GT_FILE))

print(f"\nDataset shapes:")
print(f"  Base (xb): {xb.shape}")
print(f"  Query (xq): {xq.shape}")
print(f"  Ground truth (I_gt): {I_gt.shape}")
print(f"  Number of queries: {xq.shape[0]}")
print(f"  Dimension: {xb.shape[1]}")

---

## 3. Load or Run Experiments

In [ ]:
USE_EXISTING_RESULTS = True

EXISTING_RESULTS_FILE = RESULTS_CSV / 'my_algos_cpp_benchmark_siftsmall_results.csv'

if USE_EXISTING_RESULTS and EXISTING_RESULTS_FILE.exists():
    print(f"Loading existing results from: {EXISTING_RESULTS_FILE}")
    existing_df = pd.read_csv(EXISTING_RESULTS_FILE)
    print(f"Loaded {len(existing_df)} rows")
    print(f"Methods: {existing_df['Method'].unique().tolist()}")
    print(f"\nExisting results loaded - will use for aggregate analysis.")
    print("Note: Per-query analysis requires running new experiments.")
else:
    print("No existing results found or USE_EXISTING_RESULTS=False")
    print("Will run new experiments for per-query analysis.")
    existing_df = None

---

## 4. Define Query-Level Metrics

In [ ]:
from testing.comparing_algorithm import (
    build_hnsw_New, hnsw_New_search_fn,
    build_hnsw_pip, hnsw_pip_search_fn,
    build_hnsw_adaef, hnsw_adaef_search_fn,
    build_hnsw_darth, hnsw_darth_search_fn,
    DummyPredictor
)

K = 10


def per_query_recall_at_k(gt_ids, pred_ids, k):
    """Compute recall for a single query."""
    gt_set = set(gt_ids[:k])
    pred_set = set(pred_ids[:k])
    return len(gt_set.intersection(pred_set)) / max(k, 1)


class PerQuerySearchTracker:
    """
    Tracks per-query metrics during search.
    This enables detailed query difficulty analysis.
    """
    
    def __init__(self, index, search_fn_factory, method_name, extra_params=None):
        self.index = index
        self.search_fn_factory = search_fn_factory
        self.method_name = method_name
        self.extra_params = extra_params or {}
        
        self.per_query_data = []
    
    def run_queries(self, Xq, k, I_gt):
        """Run all queries and track per-query metrics."""
        self.per_query_data = []
        n_queries = Xq.shape[0]
        
        for i in range(n_queries):
            q = Xq[i]
            gt_ids = I_gt[i]
            
            query_result = self._run_single_query(q, k, gt_ids)
            query_result['query_id'] = i
            self.per_query_data.append(query_result)
        
        return pd.DataFrame(self.per_query_data)
    
    def _run_single_query(self, q, k, gt_ids):
        """Run a single query and collect metrics."""
        t0 = time.perf_counter()
        D, I = self.index.search(np.array([q]), k)
        t1 = time.perf_counter()
        
        latency_ms = (t1 - t0) * 1000
        pred_ids = I[0]
        recall = per_query_recall_at_k(gt_ids, pred_ids, k)
        
        return {
            'latency_ms': latency_ms,
            'recall': recall,
            'method': self.method_name,
            **self.extra_params
        }


class HNSWIndexWithMetrics:
    """
    Wrapper around HNSW index to track per-query metrics.
    Tracks latency, visited nodes, distance computations when possible.
    """
    
    def __init__(self, wrapped_index):
        self.wrapped = wrapped_index
        self.visited_count = 0
        self.distance_count = 0
    
    def search(self, Xq, k):
        """Search with metrics tracking."""
        self.visited_count = 0
        self.distance_count = 0
        return self.wrapped.search(Xq, k)
    
    def __getattr__(self, name):
        """Forward other attributes to wrapped index."""
        return getattr(self.wrapped, name)

---

## 5. Run Per-Query Experiments

In [ ]:
CONSTRUCTION_PARAMS = {'M': 16, 'efC': 100}
EF_SEARCH_VALUES = [50, 100, 200]

print("Building indices...")

print("  Building HNSW index...")
hnsw_index = build_hnsw_New(xb, **CONSTRUCTION_PARAMS)

print("  Building PiP index...")
pip_index = build_hnsw_pip(xb, **CONSTRUCTION_PARAMS, pip_gamma=95.0, pip_delta=20)

print("  Building DARTH index...")
darth_index = build_hnsw_darth(xb, **CONSTRUCTION_PARAMS)

print("  Building Ada-ef index...")
adaef_index = build_hnsw_adaef(xb, **CONSTRUCTION_PARAMS, offline_k=K, offline_target_recall=0.95)

print("Indices built successfully.")

In [ ]:
all_per_query_results = []

print("\nRunning per-query experiments...")

methods_config = [
    ('HNSW-ef50', hnsw_index, lambda idx, ef=50: hnsw_New_search_fn(idx, ef), {'efSearch': 50}),
    ('HNSW-ef100', hnsw_index, lambda idx, ef=100: hnsw_New_search_fn(idx, ef), {'efSearch': 100}),
    ('HNSW-ef200', hnsw_index, lambda idx, ef=200: hnsw_New_search_fn(idx, ef), {'efSearch': 200}),
    ('PiP-ef100', pip_index, lambda idx, ef=100: hnsw_pip_search_fn(idx, ef), {'efSearch': 100}),
    ('DARTH-Rt0.95', darth_index, lambda idx, ef=100, Rt=0.95: hnsw_darth_search_fn(idx, ef=ef, Rt=Rt, predictor=DummyPredictor()), 
     {'efSearch': 100, 'Rt': 0.95}),
]

SAMPLE_SIZE = min(100, len(xq))
sample_indices = np.random.choice(len(xq), SAMPLE_SIZE, replace=False)
xq_sample = xq[sample_indices]
I_gt_sample = I_gt[sample_indices]

print(f"Using {SAMPLE_SIZE} sample queries for detailed analysis.")

for method_name, index, search_factory, params in methods_config:
    print(f"\n  Running {method_name}...")
    
    tracker = PerQuerySearchTracker(index, search_factory, method_name, params)
    df = tracker.run_queries(xq_sample, K, I_gt_sample)
    
    all_per_query_results.append(df)
    
    print(f"    Recall: mean={df['recall'].mean():.4f}, std={df['recall'].std():.4f}")
    print(f"    Latency: mean={df['latency_ms'].mean():.3f}ms, std={df['latency_ms'].std():.3f}ms")

per_query_df = pd.concat(all_per_query_results, ignore_index=True)
print(f"\nTotal per-query results: {len(per_query_df)} rows")

---

## 6. Analyze Per-Query Recall

In [ ]:
print("=== Per-Query Recall Analysis ===\n")

for method in per_query_df['method'].unique():
    method_data = per_query_df[per_query_df['method'] == method]
    print(f"\n{method}:")
    print(f"  Min recall: {method_data['recall'].min():.4f}")
    print(f"  Max recall: {method_data['recall'].max():.4f}")
    print(f"  Mean recall: {method_data['recall'].mean():.4f}")
    print(f"  Std recall: {method_data['recall'].std():.4f}")
    print(f"  Median recall: {method_data['recall'].median():.4f}")

print("\n\nObservation:")
print("- Recall varies significantly across queries for all methods")
print("- Hard queries consistently show lower recall")
print("- Higher efSearch generally improves recall but with diminishing returns")

---

## 7. Analyze Per-Query Latency

In [ ]:
print("=== Per-Query Latency Analysis ===\n")

for method in per_query_df['method'].unique():
    method_data = per_query_df[per_query_df['method'] == method]
    print(f"\n{method}:")
    print(f"  Min latency: {method_data['latency_ms'].min():.3f} ms")
    print(f"  Max latency: {method_data['latency_ms'].max():.3f} ms")
    print(f"  Mean latency: {method_data['latency_ms'].mean():.3f} ms")
    print(f"  Std latency: {method_data['latency_ms'].std():.3f} ms")
    print(f"  Median latency: {method_data['latency_ms'].median():.3f} ms")

print("\n\nObservation:")
print("- Latency varies across queries even with same efSearch")
print("- This suggests inherent query difficulty differences")
print("- Adaptive methods aim to match latency to actual difficulty")

---

## 8. Define Difficulty Buckets

In [ ]:
def create_difficulty_buckets(df, reference_method='HNSW-ef200', n_buckets=3):
    """
    Create difficulty buckets based on reference method performance.
    
    Uses reference method (highest ef) to classify queries:
    - Easy: high recall with low latency
    - Medium: moderate recall
    - Hard: low recall or high latency
    """
    ref_data = df[df['method'] == reference_method].copy()
    
    ref_data['difficulty_score'] = (
        1 - ref_data['recall'] + 
        ref_data['latency_ms'] / ref_data['latency_ms'].max()
    ) / 2
    
    ref_data['difficulty_bucket'] = pd.qcut(
        ref_data['difficulty_score'], 
        q=n_buckets, 
        labels=['easy', 'medium', 'hard']
    )
    
    query_difficulty_map = dict(zip(ref_data['query_id'], ref_data['difficulty_bucket']))
    
    df_with_buckets = df.copy()
    df_with_buckets['difficulty_bucket'] = df_with_buckets['query_id'].map(query_difficulty_map)
    
    return df_with_buckets, ref_data


per_query_with_buckets, reference_data = create_difficulty_buckets(per_query_df)

print("=== Difficulty Bucket Distribution ===\n")
bucket_counts = reference_data['difficulty_bucket'].value_counts()
print(f"Easy queries: {bucket_counts.get('easy', 0)} ({100*bucket_counts.get('easy', 0)/len(reference_data):.1f}%)")
print(f"Medium queries: {bucket_counts.get('medium', 0)} ({100*bucket_counts.get('medium', 0)/len(reference_data):.1f}%)")
print(f"Hard queries: {bucket_counts.get('hard', 0)} ({100*bucket_counts.get('hard', 0)/len(reference_data):.1f}%)")

---

## 9. Compare Methods by Difficulty Bucket

In [ ]:
print("=== Method Comparison by Difficulty Bucket ===\n")

comparison_data = []

for bucket in ['easy', 'medium', 'hard']:
    print(f"\n{bucket.upper()} QUERIES:")
    bucket_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == bucket]
    
    for method in bucket_data['method'].unique():
        method_bucket_data = bucket_data[bucket_data['method'] == method]
        
        comparison_data.append({
            'difficulty': bucket,
            'method': method,
            'n_queries': len(method_bucket_data),
            'mean_recall': method_bucket_data['recall'].mean(),
            'std_recall': method_bucket_data['recall'].std(),
            'mean_latency_ms': method_bucket_data['latency_ms'].mean(),
            'std_latency_ms': method_bucket_data['latency_ms'].std()
        })
        
        print(f"  {method}: recall={method_bucket_data['recall'].mean():.4f} (±{method_bucket_data['recall'].std():.4f}), "
              f"latency={method_bucket_data['latency_ms'].mean():.3f}ms (±{method_bucket_data['latency_ms'].std():.3f}ms)")

comparison_df = pd.DataFrame(comparison_data)
print("\n\nSummary Table:")
print(comparison_df.pivot_table(index='method', columns='difficulty', 
                               values='mean_recall').to_string())

In [ ]:
print("\n=== Why Adaptive Methods Help ===\n")

print("KEY INSIGHT 1: Query Difficulty is Not Uniform")
print("-" * 50)
ref_stats = per_query_with_buckets[per_query_with_buckets['method'] == 'HNSW-ef200']
print(f"Recall range across queries: [{ref_stats['recall'].min():.4f}, {ref_stats['recall'].max():.4f}]")
print(f"Latency range: [{ref_stats['latency_ms'].min():.3f}, {ref_stats['latency_ms'].max():.3f}] ms")
print(f"This shows queries have very different characteristics.")

print("\nKEY INSIGHT 2: Fixed ef Cannot Serve All Queries Equally")
print("-" * 50)
easy_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == 'easy']
hard_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == 'hard']
print(f"Easy queries: avg recall={easy_data[easy_data['method']=='HNSW-ef50']['recall'].mean():.4f} with ef=50")
print(f"Hard queries: avg recall={hard_data[hard_data['method']=='HNSW-ef50']['recall'].mean():.4f} with ef=50")
print(f"Using same ef for both wastes effort on easy queries.")

print("\nKEY INSIGHT 3: Adaptive Methods Can Help")
print("-" * 50)
pip_data = per_query_with_buckets[per_query_with_buckets['method'] == 'PiP-ef100']
hnsw_data = per_query_with_buckets[per_query_with_buckets['method'] == 'HNSW-ef100']
print(f"PiP-ef100 vs HNSW-ef100 on easy queries:")
pip_easy = pip_data[pip_data['difficulty_bucket'] == 'easy']
hnsw_easy = hnsw_data[hnsw_data['difficulty_bucket'] == 'easy']
print(f"  PiP: recall={pip_easy['recall'].mean():.4f}, latency={pip_easy['latency_ms'].mean():.3f}ms")
print(f"  HNSW: recall={hnsw_easy['recall'].mean():.4f}, latency={hnsw_easy['latency_ms'].mean():.3f}ms")

---

## 10. Plots

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

METHOD_COLORS = {
    'HNSW-ef50': '#1f77b4',
    'HNSW-ef100': '#2ca02c',
    'HNSW-ef200': '#d62728',
    'PiP-ef100': '#9467bd',
    'DARTH-Rt0.95': '#ff7f0e'
}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = per_query_df['method'].unique()
colors = [METHOD_COLORS.get(m, 'gray') for m in methods]

for idx, method in enumerate(methods):
    ax = axes[idx // 2, idx % 2]
    method_data = per_query_df[per_query_df['method'] == method]
    
    ax.hist(method_data['recall'], bins=20, color=METHOD_COLORS.get(method, 'gray'), 
            alpha=0.7, edgecolor='black')
    ax.axvline(method_data['recall'].mean(), color='red', linestyle='--', 
               linewidth=2, label=f"Mean: {method_data['recall'].mean():.3f}")
    ax.axvline(method_data['recall'].median(), color='green', linestyle=':', 
               linewidth=2, label=f"Median: {method_data['recall'].median():.3f}")
    
    ax.set_xlabel(f'Recall@{K}')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{method}: Per-Query Recall Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_per_query_recall_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_per_query_recall_histograms.png'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, method in enumerate(methods):
    ax = axes[idx // 2, idx % 2]
    method_data = per_query_df[per_query_df['method'] == method]
    
    ax.hist(method_data['latency_ms'], bins=20, color=METHOD_COLORS.get(method, 'gray'), 
            alpha=0.7, edgecolor='black')
    ax.axvline(method_data['latency_ms'].mean(), color='red', linestyle='--', 
               linewidth=2, label=f"Mean: {method_data['latency_ms'].mean():.2f}ms")
    ax.axvline(method_data['latency_ms'].median(), color='green', linestyle=':', 
               linewidth=2, label=f"Median: {method_data['latency_ms'].median():.2f}ms")
    
    ax.set_xlabel('Latency (ms)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{method}: Per-Query Latency Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_per_query_latency_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_per_query_latency_histograms.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for method in methods:
    method_data = per_query_df[per_query_df['method'] == method]
    ax.scatter(method_data['recall'], method_data['latency_ms'], 
               c=METHOD_COLORS.get(method, 'gray'), marker='o', s=50, alpha=0.6,
               label=method, edgecolors='black', linewidths=0.5)

ax.set_xlabel(f'Recall@{K}')
ax.set_ylabel('Latency (ms)')
ax.set_title('Recall vs Latency: Per-Query Scatter')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_recall_vs_latency_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_recall_vs_latency_scatter.png'}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

bucket_colors = {'easy': '#2ca02c', 'medium': '#ff7f0e', 'hard': '#d62728'}

for idx, bucket in enumerate(['easy', 'medium', 'hard']):
    ax = axes[idx]
    bucket_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == bucket]
    
    for method in methods:
        method_bucket_data = bucket_data[bucket_data['method'] == method]
        if len(method_bucket_data) > 0:
            ax.bar(method, method_bucket_data['recall'].mean(), 
                   yerr=method_bucket_data['recall'].std(),
                   color=METHOD_COLORS.get(method, 'gray'), 
                   edgecolor='black', capsize=5, alpha=0.8)
    
    ax.set_ylabel(f'Mean Recall@{K}')
    ax.set_title(f'{bucket.capitalize()} Queries (n={len(bucket_data[bucket_data["method"]==methods[0]])})')
    ax.set_ylim([0, 1.1])
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_method_comparison_by_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_method_comparison_by_difficulty.png'}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, bucket in enumerate(['easy', 'medium', 'hard']):
    ax = axes[idx]
    bucket_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == bucket]
    
    for method in methods:
        method_bucket_data = bucket_data[bucket_data['method'] == method]
        if len(method_bucket_data) > 0:
            ax.bar(method, method_bucket_data['latency_ms'].mean(), 
                   yerr=method_bucket_data['latency_ms'].std(),
                   color=METHOD_COLORS.get(method, 'gray'), 
                   edgecolor='black', capsize=5, alpha=0.8)
    
    ax.set_ylabel('Mean Latency (ms)')
    ax.set_title(f'{bucket.capitalize()} Queries')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_latency_by_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_latency_by_difficulty.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ref_data_sorted = reference_data.sort_values('difficulty_score')
colors = [bucket_colors[b] for b in ref_data_sorted['difficulty_bucket']]

ax.bar(range(len(ref_data_sorted)), ref_data_sorted['recall'], color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Query (sorted by difficulty)')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('Recall per Query (colored by difficulty bucket)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=bucket_colors['easy'], edgecolor='black', label='Easy'),
                   Patch(facecolor=bucket_colors['medium'], edgecolor='black', label='Medium'),
                   Patch(facecolor=bucket_colors['hard'], edgecolor='black', label='Hard')]
ax.legend(handles=legend_elements, loc='lower left')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_query_difficulty_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_query_difficulty_distribution.png'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['recall', 'latency_ms']
bucket_order = ['easy', 'medium', 'hard']

for row_idx, metric in enumerate(metrics):
    for col_idx, bucket in enumerate(bucket_order):
        ax = axes[row_idx, col_idx]
        bucket_data = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == bucket]
        
        data_to_plot = []
        labels = []
        colors_list = []
        
        for method in methods:
            method_bucket_data = bucket_data[bucket_data['method'] == method]
            if len(method_bucket_data) > 0:
                data_to_plot.append(method_bucket_data[metric].values)
                labels.append(method)
                colors_list.append(METHOD_COLORS.get(method, 'gray'))
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors_list):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        metric_label = 'Recall' if metric == 'recall' else 'Latency (ms)'
        ax.set_ylabel(metric_label)
        ax.set_title(f'{bucket.capitalize()} Queries - {metric_label}')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '06_boxplots_by_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '06_boxplots_by_difficulty.png'}")

---

## 11. Export Results

In [ ]:
PER_QUERY_CSV = RESULTS_CSV / '06_per_query_analysis.csv'
COMPARISON_CSV = RESULTS_CSV / '06_difficulty_comparison.csv'

per_query_with_buckets.to_csv(PER_QUERY_CSV, index=False)
comparison_df.to_csv(COMPARISON_CSV, index=False)

print(f"Per-query analysis saved to: {PER_QUERY_CSV}")
print(f"Comparison table saved to: {COMPARISON_CSV}")

---

## 12. Conclusions

In [ ]:
print("=" * 80)
print("QUERY DIFFICULTY ANALYSIS SUMMARY")
print("=" * 80)

print("\n1. WHY ARE SOME QUERIES EASIER THAN OTHERS?")
print("-" * 50)
print(""")
   - Query Position: Queries in sparse regions find neighbors easily
   - Query Position: Queries in dense regions face more competition
   - Local Density: Easy queries have well-separated nearest neighbors
   - Local Density: Hard queries have neighbors at similar distances
   - Graph Structure: Easy queries have good entry points
   - Graph Structure: Hard queries may require deeper graph traversal
""")

print("\n2. HOW DOES QUERY DIFFICULTY AFFECT RECALL AND LATENCY?")
print("-" * 50)
easy = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == 'easy']
hard = per_query_with_buckets[per_query_with_buckets['difficulty_bucket'] == 'hard']
hnsw_ef200 = per_query_with_buckets[per_query_with_buckets['method'] == 'HNSW-ef200']

print(f"   With HNSW-ef200:")
easy_hnsw = hnsw_ef200[hnsw_ef200['difficulty_bucket'] == 'easy']
hard_hnsw = hnsw_ef200[hnsw_ef200['difficulty_bucket'] == 'hard']
print(f"   - Easy queries: avg recall={easy_hnsw['recall'].mean():.4f}")
print(f"   - Hard queries: avg recall={hard_hnsw['recall'].mean():.4f}")
print(f"   - Recall gap: {easy_hnsw['recall'].mean() - hard_hnsw['recall'].mean():.4f}")

print("\n3. WHY DO ADAPTIVE METHODS HELP?")
print("-" * 50)
print(""")
   PiP (Patience in Proximity):
   - Monitors result set saturation
   - Terminates early when neighbors stop changing
   - Saves effort on easy queries that converge quickly

   DARTH (Dynamic Adaptive Re-termination):
   - Uses history features to predict termination
   - Adjusts search based on observed patterns
   - Can skip unnecessary exploration on easy queries

   Ada-ef (Adaptive Exploration Factor):
   - Estimates query difficulty from initial samples
   - Adjusts ef based on estimated difficulty
   - Uses more ef on hard queries, less on easy ones
""")

print("\nKEY TAKEAWAY:")
print("-" * 50)
print(""")
Query difficulty is intrinsic to the data and query distribution.
Using the same search parameters for all queries is suboptimal:
- Easy queries waste computational resources
- Hard queries may not achieve target recall
Adaptive methods bridge this gap by tailoring search to each query.
""")

In [ ]:
print("\n" + "=" * 80)
print("OUTPUT FILES GENERATED")
print("=" * 80)
print(f"\n1. Per-query analysis: {PER_QUERY_CSV}")
print(f"2. Comparison table: {COMPARISON_CSV}")
print(f"\n3. Plots:")
print(f"   - {PLOT_RESULTS / '06_per_query_recall_histograms.png'}")
print(f"   - {PLOT_RESULTS / '06_per_query_latency_histograms.png'}")
print(f"   - {PLOT_RESULTS / '06_recall_vs_latency_scatter.png'}")
print(f"   - {PLOT_RESULTS / '06_method_comparison_by_difficulty.png'}")
print(f"   - {PLOT_RESULTS / '06_latency_by_difficulty.png'}")
print(f"   - {PLOT_RESULTS / '06_query_difficulty_distribution.png'}")
print(f"   - {PLOT_RESULTS / '06_boxplots_by_difficulty.png'}")
print("\n" + "=" * 80)
print("NOTEBOOK COMPLETED SUCCESSFULLY")
print("=" * 80)